## Lecture 4 Clustering 

Unsupervised learning: Clustering

* No labels to determine the clusters
* Data is clustered by variable similarity instead
* Using K-means clustering
* We will create financial solvency clusters based on ROA and leverage

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
# Load the dataset (we will use a subset for 2D clustering)
df_clus = pd.DataFrame(pd.read_csv('/home/jovyan/shared/Data/ml_data.csv')[['net_profit_over_total_assets', 'book_value_of_equity_over_total_liabilities', 'class']])
df_clus = df_clus.rename(columns={'net_profit_over_total_assets': 'ROA', 'book_value_of_equity_over_total_liabilities': 'Leverage'})
df_clus

In [ ]:
df_clus.info()

In [ ]:
df_clus.replace("?", np.nan, inplace=True)
df_clus.dropna(inplace=True)
df_clus[["ROA", "Leverage"]] = df_clus[["ROA", "Leverage"]].astype('float64')
df_clus.info()

In [ ]:
# Winsorize / clip outliers to remove extreme values (1% and 99% quantiles)
df_clus = df_clus.clip(lower=df_clus.quantile(0.01), upper=df_clus.quantile(0.99), axis=1)

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_clus[['ROA', 'Leverage']])

# Put scaled features back into a DataFrame
scaled_df = pd.DataFrame(scaled_features, columns=['ROA_scaled', 'Leverage_scaled'])
scaled_df['class'] = df_clus['class']
df_clus.head()

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(df_clus['Leverage'][df_clus['class'] == 0], 
            df_clus['ROA'][df_clus['class'] == 0], 
            color='#3498db', alpha=0.5, label='Healthy (Non-Bankrupt)', edgecolor='k', s=40)
plt.scatter(df_clus['Leverage'][df_clus['class'] == 1], 
            df_clus['ROA'][df_clus['class'] == 1], 
            color='#e74c3c', alpha=0.9, label='Bankrupt', marker='X', s=100)
plt.xlabel('Leverage (Equity / Liabilities)')
plt.ylabel('ROA (Net Profit / Total Assets)')
plt.title('Corporate Financial Space: ROA vs Leverage')
plt.legend()
plt.show()

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'   # Prevents a thread memory leak warning in KMeans on Windows

# Fit KMeans clustering with K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(scaled_df[['ROA_scaled', 'Leverage_scaled']])

# Add cluster labels to our dataframes
df_clus['clusters'] = kmeans.labels_
scaled_df['clusters'] = kmeans.labels_

In [ ]:
centers = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers)

plt.figure(figsize=(10, 8))
colors_list = ['#2ecc71', '#3498db', '#e74c3c']
for i in range(3):
    plt.scatter(df_clus['Leverage'][df_clus['clusters'] == i],
                df_clus['ROA'][df_clus['clusters'] == i],
                color=colors_list[i], alpha=0.5, label=f'Cluster {i}', edgecolor='k', s=40)

plt.scatter(centers_original[:, 1], centers_original[:, 0], 
            s=250, color='yellow', marker='*', edgecolors='black', linewidths=2, label='Centroids')

plt.xlabel('Leverage (Equity / Liabilities)')
plt.ylabel('ROA (Net profit / Total assets)')
plt.title('K-Means Clustering of Companies (K = 3)')
plt.legend()
plt.show()

In [ ]:
cluster_profile = df_clus.groupby('clusters').agg({
    'ROA': 'mean',
    'Leverage': 'mean',
    'class': ['mean', 'count']
})
cluster_profile.columns = ['Mean ROA', 'Mean Leverage', 'Bankruptcy Rate', 'Company Count']
cluster_profile